# sshPortfolio LLM 파인튜닝 (QLoRA) - Colab 노트북

포트폴리오 챗봇의 답변 규칙(근거 기반 답변, 모름 처리, `관련:`/`다음:` 형식)을 공개 소형 모델(**Qwen/Qwen3-1.7B**)에 QLoRA로 추가 학습하고, **같은 기반 모델의 학습 전후**를 비교합니다.
운영 중인 Gemini 챗봇은 건드리지 않습니다.

**위에서부터 순서대로** 실행하세요. 각 셀은 실패하면 이유와 다음에 할 일을 출력하고 멈춥니다.

| 단계 | 셀 |
| --- | --- |
| 준비 | 1 GPU 확인 → 2 Google Drive 연결 → 3 코드 가져오기 → 4 라이브러리 설치·버전 확인 → 5 코드 버전 기록 |
| 데이터 | 6 데이터 검증·분할 → 7 토큰 길이·마스크 검사(모델 커밋 고정) |
| 학습 | 8 학습 전 소수 질문 추론 → 9 smoke test(5스텝) → 10 소규모 전체 학습 |
| 평가 | 11 학습 전후 생성 → 12 비교 보고서 |
| 복구 | 13 어댑터 다시 불러오기 예시 · 연결이 끊겼을 때 |

> **노트북 저장 ≠ 결과 저장.** 노트북 파일(.ipynb)을 저장해도 학습 결과(어댑터, 체크포인트, 평가 결과)는 저장되지 않습니다. 결과는 모두 Google Drive의 `MyDrive/sshportfolio-llm/` 아래에 쓰도록 되어 있습니다. Colab 런타임 디스크(`/content`)는 연결이 끊기면 사라집니다.

> 무료 Colab은 GPU 종류·메모리·사용 시간이 보장되지 않습니다. 학습 중 연결이 끊겨도 Drive의 체크포인트에서 이어서 할 수 있습니다(맨 아래 '연결이 끊겼을 때' 참고).

모델 다운로드(약 4GB)와 학습은 Colab 안에서 이루어집니다. 비용이 드는 작업(유료 GPU, 모델 업로드)은 이 노트북에 없습니다.

## 1. GPU 확인
메뉴 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: GPU(T4 등)** 를 고르고 저장한 뒤 실행합니다. GPU가 없으면 여기서 멈춥니다.

In [ ]:
import subprocess, sys
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "nvidia-smi 없음")
import torch
print("Python", sys.version.split()[0], "| torch", torch.__version__, "| CUDA", torch.version.cuda)
if not torch.cuda.is_available():
    raise SystemExit("GPU 런타임이 아닙니다. [런타임] → [런타임 유형 변경] 에서 GPU 를 고른 뒤 이 셀부터 다시 실행하세요.")
p = torch.cuda.get_device_properties(0)
major, minor = torch.cuda.get_device_capability(0)
print(f"GPU: {p.name} | VRAM {p.total_memory / 1024**3:.1f} GiB | compute capability {major}.{minor} | bf16 {'지원' if major >= 8 else '미지원 → fp16 사용'}")

## 2. Google Drive 연결 (결과 저장 위치)
결과는 `DRIVE_ROOT` 아래에 저장됩니다. 처음 실행하면 Google 계정 접근 허용 창이 뜹니다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/sshportfolio-llm"   # configs/experiment.json 의 output.colab_drive_root 와 같아야 함
CODE_DIR = "/content/sshPortfolio"
import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
print("결과 저장 위치:", DRIVE_ROOT)
print("기존 내용:", os.listdir(DRIVE_ROOT))

## 3. 코드 가져오기
아직 커밋·푸시하지 않은 실험 파일을 쓰려면 **ZIP 방식(기본)** 을 씁니다.

1. Mac 터미널에서 저장소 루트로 이동해 `python3 llm/scripts/package_for_colab.py` 를 실행합니다 → `llm/dist/sshportfolio-llm.zip` 생성
2. 아래 셀을 실행하면 업로드 버튼이 나옵니다. ZIP을 고르면 Drive(`DRIVE_ROOT/sshportfolio-llm.zip`)에도 복사해 두므로, 다음 런타임부터는 다시 올리지 않아도 됩니다.
   - 데이터나 코드를 고친 뒤에는 ZIP을 다시 만들고 `UPLOAD_NEW_ZIP = True` 로 실행하세요.

GitHub에 푸시한 브랜치를 쓰려면 `CODE_SOURCE = "git"` 으로 바꾸고 `GIT_BRANCH` 를 적습니다. 기본 브랜치에 `llm/` 이 있다고 가정하지 않습니다.

In [ ]:
CODE_SOURCE = "zip"          # "zip" 또는 "git"
UPLOAD_NEW_ZIP = False       # True 면 Drive 에 저장된 ZIP 대신 새로 업로드
GIT_URL = "https://github.com/simsohyeon/sshPortfolio.git"
GIT_BRANCH = "main"          # llm/ 이 푸시된 브랜치 이름

import os, shutil, subprocess, zipfile
ZIP_IN_DRIVE = f"{DRIVE_ROOT}/sshportfolio-llm.zip"
if os.path.exists(CODE_DIR):
    shutil.rmtree(CODE_DIR)
if CODE_SOURCE == "zip":
    if UPLOAD_NEW_ZIP or not os.path.exists(ZIP_IN_DRIVE):
        from google.colab import files
        up = files.upload()
        if not up:
            raise SystemExit("ZIP 을 업로드하지 않았습니다.")
        name = next(iter(up))
        shutil.copy(name, ZIP_IN_DRIVE)
        print("Drive 에 저장:", ZIP_IN_DRIVE)
    with zipfile.ZipFile(ZIP_IN_DRIVE) as z:
        z.extractall(CODE_DIR)
elif CODE_SOURCE == "git":
    subprocess.run(["git", "clone", "--depth", "1", "-b", GIT_BRANCH, GIT_URL, CODE_DIR], check=True)
else:
    raise SystemExit("CODE_SOURCE 는 zip 또는 git")
if not os.path.exists(f"{CODE_DIR}/llm/train.py"):
    raise SystemExit("llm/train.py 가 없습니다. ZIP 을 다시 만들었는지, 또는 브랜치에 llm/ 이 푸시됐는지 확인하세요.")
os.chdir(CODE_DIR)
print("코드 위치:", os.getcwd())
print(open("llm/VERSION.json").read() if os.path.exists("llm/VERSION.json") else "(VERSION.json 없음 - git 방식)")

## 4. 라이브러리 설치와 버전 확인
`llm/requirements-colab.txt` 의 고정 버전을 설치합니다. torch는 Colab에 설치된 것을 그대로 씁니다.
설치 후 아래 확인 셀에서 **불일치가 보이면** 메뉴 **런타임 → 세션 다시 시작** 후 1~3번 셀을 다시 실행하고 이 셀은 건너뛰어도 됩니다.

In [ ]:
!pip install -q -r llm/requirements-colab.txt

In [ ]:
import subprocess, sys, re
want = dict(re.findall(r"^([A-Za-z0-9_.-]+)==([^\s#]+)", open("llm/requirements-colab.txt").read(), re.M))
# 새 파이썬 프로세스로 실제 설치된 버전을 확인 (이 커널에 이미 올라간 옛 버전과 구분)
out = subprocess.run([sys.executable, "-c", "import importlib.metadata as m, json, sys; print(json.dumps({k: m.version(k) for k in sys.argv[1:]}))", *want],
                     capture_output=True, text=True, check=True).stdout
import json
have = json.loads(out)
bad = {k: (v, have.get(k)) for k, v in want.items() if have.get(k) != v}
for k in want:
    print(f"{k:<14} 요구 {want[k]:<10} 설치 {have.get(k)}")
import torch
print("torch", torch.__version__)
if bad:
    raise SystemExit(f"버전 불일치: {bad}")
import transformers, sys as _s
if transformers.__version__ != want["transformers"]:
    print("⚠️ 이 커널에는 이전 transformers 가 올라가 있습니다. [런타임] → [세션 다시 시작] 후 1~3번 셀을 다시 실행하세요.")
else:
    print("OK")

## 5. 코드 버전 기록
학습 기록(`run_info.json`)에 자동으로 남는 값과 같습니다. ZIP 방식이면 ZIP을 만들 때의 git 커밋·미커밋 여부·`llm/` 내용 해시가 나옵니다.

In [ ]:
import sys, json
sys.path.insert(0, "llm")
from common import code_version
print(json.dumps(code_version(), indent=2, ensure_ascii=False))

## 6. 데이터 검증과 분할
- `validate_dataset.py`: JSONL·필수 필드·출처·프로젝트 ID·형식·중복을 검사합니다.
- `prepare_dataset.py`: **검토 완료(`reviewed`) 항목만** 학습·검증·평가로 나눕니다(같은 seed면 같은 결과). 검토 완료가 30건 미만이면 분할하지 않고 검토할 항목 목록을 보여 줍니다 → Mac에서 검토 후 ZIP을 다시 만들어 3번 셀부터 다시 실행하세요.

In [ ]:
!python llm/scripts/validate_dataset.py

In [ ]:
import subprocess
r = subprocess.run(["python", "llm/scripts/prepare_dataset.py"])
if r.returncode == 2:
    raise SystemExit("검토 완료 데이터가 부족합니다. llm/data/README.md 의 검토 절차를 진행한 뒤 다시 실행하세요.")
if r.returncode != 0:
    raise SystemExit("분할 실패 - 위 오류를 확인하세요.")

## 7. 토큰 길이·마스크 검사 (모델 커밋 고정)
- 처음 실행할 때 Hugging Face에서 `Qwen/Qwen3-1.7B` 의 **현재 커밋 SHA와 라이선스**를 조회해 `DRIVE_ROOT/<experiment>/model.json` 에 고정합니다. 이후 모든 단계가 이 커밋을 씁니다.
- 실제 토크나이저로 문맥+질문+답변 길이를 재고 `max_length` 를 정합니다. 상한을 넘으면 **자르지 않고** 멈춥니다.
- assistant 정답 토큰에만 loss가 걸리는지 검사하고, 학습 대상 텍스트 예시를 출력합니다(`<think>` 빈 블록 + 정답 + `<|im_end|>` 여야 함).

In [ ]:
!python llm/train.py --check-only --output-root "$DRIVE_ROOT"

## 8. 학습 전 소수 질문 추론
기반 모델이 지금 어떻게 답하는지 평가셋 앞 3개로 확인합니다. (4-bit 로드, greedy 생성)

In [ ]:
EXP = json.load(open("llm/configs/experiment.json"))["experiment_name"]
RUN_FULL = f"{DRIVE_ROOT}/{EXP}/full"
RUN_SMOKE = f"{DRIVE_ROOT}/{EXP}/smoke"
print(RUN_FULL)
!python llm/evaluate.py generate --run-dir "$RUN_FULL" --tag base --limit 3 --show

## 9. smoke test (5스텝)
학습 경로 전체(4-bit 로드 → LoRA → 실제 배치 라벨 검사 → 학습 → 저장)를 짧게 확인합니다. 결과는 `.../smoke` 에 따로 저장됩니다.
여기서 OOM이 나면 `llm/README.md` 의 **OOM·길이 초과** 절차를 따릅니다.

In [ ]:
!python llm/train.py --smoke --output-root "$DRIVE_ROOT"

## 10. 소규모 전체 학습
`configs/experiment.json` 의 설정(기본 2 epoch, LoRA r=8, lr 1e-4)으로 학습합니다. 체크포인트는 `save_steps` 마다 Drive에 저장되고,
**같은 셀을 다시 실행하면 마지막 체크포인트에서 이어서** 학습합니다. 끝나면 `run_info.json` 에 학습 시간·peak VRAM·loss가 남습니다.

In [ ]:
!python llm/train.py --output-root "$DRIVE_ROOT"
!cat "$RUN_FULL/run_info.json" | head -c 3000

## 11. 학습 전후 생성 (같은 평가셋·같은 설정)
평가셋(test) 전체에 대해 학습 전(기반 모델)과 학습 후(기반 모델 + 어댑터)를 각각 생성합니다. 이미 만든 결과는 건너뛰므로 끊긴 뒤 다시 실행해도 됩니다.
설정 조정은 `--split validation` 으로 검증셋에서 하고, test는 마지막 확인에만 씁니다.

In [ ]:
!python llm/evaluate.py generate --run-dir "$RUN_FULL" --tag base
!python llm/evaluate.py generate --run-dir "$RUN_FULL" --tag tuned --adapter "$RUN_FULL/adapter"

## 12. 비교 보고서
형식 검사 집계와 질문별 나란히 보기, 사람 평가 칸이 있는 Markdown 보고서를 만듭니다. Drive의 `.../full/eval/` 에서 내려받아 원문과 대조하며 채우세요.
자동 검사는 **형식**만 봅니다. 내용 정확성은 사람 평가로 확인해야 합니다.

In [ ]:
!python llm/evaluate.py compare --before "$RUN_FULL/eval/test-base.jsonl" --after "$RUN_FULL/eval/test-tuned.jsonl"
!ls -la "$RUN_FULL/eval"

## 13. 어댑터 다시 불러오기 (새 런타임에서)
어댑터(`adapter/`)는 LoRA 가중치뿐이라 **혼자서는 실행되지 않습니다.** 학습 때와 **같은 기반 모델·같은 커밋(`model.json`)** 을 먼저 불러온 뒤 그 위에 붙입니다.
새 런타임에서는 1~4번 셀을 실행한 뒤 이 셀을 실행하면 됩니다.

In [ ]:
import json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

pinned = json.load(open(f"{DRIVE_ROOT}/{EXP}/model.json"))          # 기반 모델 ID·커밋
bf16 = torch.cuda.get_device_capability(0)[0] >= 8
dtype = torch.bfloat16 if bf16 else torch.float16
base = AutoModelForCausalLM.from_pretrained(
    pinned["id"], revision=pinned["revision"], dtype=dtype, device_map={"": 0},
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                           bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype),
)
tok = AutoTokenizer.from_pretrained(pinned["id"], revision=pinned["revision"])
model = PeftModel.from_pretrained(base, f"{RUN_FULL}/adapter").eval()

sys.path.insert(0, "llm")
from common import build_messages, load_config, load_context
ctx = load_context(load_config())
msgs = build_messages("Kafka는 어디에 사용했나요?", ctx)               # 운영 챗봇과 같은 시스템 프롬프트
enc = tok.apply_chat_template(msgs, add_generation_prompt=True, enable_thinking=False,
                              tokenize=True, return_dict=True, return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(**enc, max_new_tokens=512, do_sample=False)
print(tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True))

## 연결이 끊겼을 때
1. 메뉴 **런타임 → 런타임 유형 변경**에서 GPU를 다시 고릅니다.
2. **1 → 2 → 3 → 4 → 5** 셀을 다시 실행합니다(ZIP은 Drive에 있으므로 다시 올리지 않아도 됨).
3. 6번 셀(분할)은 같은 seed로 같은 결과를 만듭니다. 다시 실행해도 됩니다.
4. 끊긴 단계의 셀을 다시 실행합니다.
   - 학습(9·10번): Drive의 `checkpoints/` 에서 자동으로 이어서 학습합니다. 이미 끝난 실행은 "이미 학습이 끝난 실행"이라고 알려 주고 넘어갑니다.
   - 평가(11번): 이미 만든 결과 파일은 건너뜁니다.

### 처음부터 다시 하고 싶을 때
`configs/experiment.json` 의 `experiment_name` 을 바꾸면 새 폴더에 따로 저장됩니다(기존 결과는 남음). Drive의 기존 폴더를 지우는 것은 직접 해야 합니다.